In [2]:
import pandas as pd
import polars as pl
import fsspec
import plotly.express as px
import ast
import json
import numpy as np

[Link al dataset](https://huggingface.co/datasets/finiteautomata/news-argentina)

In [3]:


fs, _, paths = fsspec.get_fs_token_paths("hf://datasets/finiteautomata/news-argentina/data/")
files = fs.ls(paths[0])

print(files)

[{'name': 'datasets/finiteautomata/news-argentina/data/train-00000-of-00003-0bc26f10064c1eef.parquet', 'size': 172488385, 'type': 'file', 'blob_id': '5bf3920ba46a015266d4122af4edec33e2fbdfc8', 'lfs': BlobLfsInfo(size=172488385, sha256='adfa8908ce41f906ada669d4ec5092a6663de4d168ebd1591c1b68e5461d0f0a', pointer_size=134), 'last_commit': None, 'security': None}, {'name': 'datasets/finiteautomata/news-argentina/data/train-00000-of-00003-9bb91ef98e670654.parquet', 'size': 173314710, 'type': 'file', 'blob_id': '29ad446016e4a4721a5db1f466e64adf22835659', 'lfs': BlobLfsInfo(size=173314710, sha256='e7275511ae03859daa3758efee7a8dc29557619e8d35492e0d23c81f93ff4205', pointer_size=134), 'last_commit': None, 'security': None}, {'name': 'datasets/finiteautomata/news-argentina/data/train-00001-of-00003-718f7f8b92936f2e.parquet', 'size': 210436432, 'type': 'file', 'blob_id': '9eb8f697aff3b6b83291c259a6685e749c3ebf9e', 'lfs': BlobLfsInfo(size=210436432, sha256='4843c67e07f96976bb127e7c798f085e1c05f24914

In [4]:

fs, _, paths = fsspec.get_fs_token_paths("hf://datasets/finiteautomata/news-argentina/data/")
files_info = fs.ls(paths[0])

# me quedo solo con los nombres (paths completos dentro de hf://)
files = [f"hf://{f['name']}" for f in files_info]

print(files)

['hf://datasets/finiteautomata/news-argentina/data/train-00000-of-00003-0bc26f10064c1eef.parquet', 'hf://datasets/finiteautomata/news-argentina/data/train-00000-of-00003-9bb91ef98e670654.parquet', 'hf://datasets/finiteautomata/news-argentina/data/train-00001-of-00003-718f7f8b92936f2e.parquet', 'hf://datasets/finiteautomata/news-argentina/data/train-00002-of-00003-3968892844e0509d.parquet']


In [5]:
fs, _, paths = fsspec.get_fs_token_paths("hf://datasets/finiteautomata/news-argentina/data/")
files_info = fs.ls(paths[0])

# Extraigo solo los nombres en formato completo hf://...
files = [f"hf://{f['name']}" for f in files_info if f['name'].endswith(".parquet")]

# Ordeno por nombre (para que respete el orden 00000, 00001, 00002...)
files = sorted(files)

print("Archivos encontrados:")
for f in files:
    print(f)

Archivos encontrados:
hf://datasets/finiteautomata/news-argentina/data/train-00000-of-00003-0bc26f10064c1eef.parquet
hf://datasets/finiteautomata/news-argentina/data/train-00000-of-00003-9bb91ef98e670654.parquet
hf://datasets/finiteautomata/news-argentina/data/train-00001-of-00003-718f7f8b92936f2e.parquet
hf://datasets/finiteautomata/news-argentina/data/train-00002-of-00003-3968892844e0509d.parquet


In [6]:
lf = pl.scan_parquet(
    files,
    cast_options=pl.ScanCastOptions(extra_struct_fields="ignore")
)

pdf = lf.collect()

In [7]:
df = pdf.to_pandas()

print(df.info())

df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97898 entries, 0 to 97897
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   tweet_id    97898 non-null  object
 1   text        97898 non-null  object
 2   title       89933 non-null  object
 3   url         89933 non-null  object
 4   user        97898 non-null  object
 5   body        89933 non-null  object
 6   created_at  97898 non-null  object
 7   comments    97898 non-null  object
dtypes: object(8)
memory usage: 6.0+ MB
None


,tweet_id,text,title,url,user,body,created_at,comments
0,1376941547602739202,El dólar blue cada vez más barato: cae a $ 141...,El dólar blue cada vez más barato: cayó a $ 14...,https://www.clarin.com/economia/dolar-blue-vez...,clarincom,Este martes el dólar blue retomó su tendencia ...,2021-03-30T16:57:01Z,"[{'created_at': '2021-03-30T16:58:04Z', 'text'..."
1,1376945423273840642,Venezuela le declara la guerra a Facebook por ...,Venezuela le declara la guerra a Facebook por ...,https://www.clarin.com/tecnologia/venezuela-de...,clarincom,El presidente Nicolás Maduro le declaró la gue...,2021-03-30T17:12:25Z,"[{'created_at': '2021-03-30T17:14:14Z', 'text'..."
2,1376946868161220610,Polémica: un proyecto K propone multas millona...,Polémica: un proyecto K propone multas millona...,https://www.clarin.com/politica/polemica-proye...,clarincom,Mientras el presidente Alberto Fernández decid...,2021-03-30T17:18:10Z,"[{'created_at': '2021-03-30T17:19:22Z', 'text'..."
3,1376948553977896971,"'Ser argentino' o 'dejar de serlo', la insólit...",Mendoza: preparan un insólito plebiscito para ...,https://www.clarin.com/politica/-argentino-dej...,clarincom,"A mediados de 2020, el ex gobernador Alfredo C...",2021-03-30T17:24:52Z,"[{'created_at': '2021-03-30T17:25:43Z', 'text'..."
4,1376949422941220870,"El tamaño de los penes, cada vez más chicos po...","El tamaño de los penes, cada vez más chicos po...",https://www.cronica.com.ar/info-general/Los-ta...,cronica,Una científica ambiental aseguró que los penes...,2021-03-30T17:28:19Z,"[{'created_at': '2021-03-30T17:29:50Z', 'text'..."


In [8]:
df.comments[0]

array([{'created_at': '2021-03-30T16:58:04Z', 'text': '@clarincom El dólar siempre es barato.', 'tweet_id': '1376941811785158665', 'user_id': '1314594430402416641'},
       {'created_at': '2021-03-30T16:58:43Z', 'text': '@clarincom Que lindo va estar todo cuando explote todo en las paso , deja vu', 'tweet_id': '1376941974180225025', 'user_id': '1227404911572324354'},
       {'created_at': '2021-03-30T16:58:53Z', 'text': '@clarincom No te puedo creer, que vuelva el gato!!!!', 'tweet_id': '1376942013489291266', 'user_id': '1066711563636260865'},
       {'created_at': '2021-03-30T16:59:16Z', 'text': '@clarincom Jajajjajajaa como les hicieron comprar dólares a los giles que les creen que el dólar iba a estar a 300', 'tweet_id': '1376942113011740673', 'user_id': '1361787028514430976'},
       {'created_at': '2021-03-30T17:00:33Z', 'text': '@clarincom El dolar no baja, toma impulso para volver a subir.', 'tweet_id': '1376942434203099141', 'user_id': '406937994'},
       {'created_at': '2021-

In [9]:
df.user.value_counts()

user
infobae            33895
clarincom          26181
LANACION           24505
cronica             5635
pagina12            4894
laderechadiario     2595
izquierdadiario      193
Name: count, dtype: int64

In [10]:
## Transformar Data y crear columnas complementarias
# transformar created_at a yyyy-mm-dd
df["created_at"] = pd.to_datetime(df["created_at"])



In [11]:
# cantidad de palabras en title y body
df["words_title"] = df["title"].fillna("").str.split().str.len()
df["words_body"] = df["body"].fillna("").str.split().str.len()



In [ ]:
# cantidad de comentarios por registro
#No está funcionando!!!!
## retomar

def count_comments(x):
    # Caso None explícito
    if x is None:
        return 0

    # Caso numpy array
    if isinstance(x, np.ndarray):
        return len(x)

    # Caso lista o tupla
    if isinstance(x, (list, tuple)):
        return len(x)

    # Caso dict -> un comentario
    if isinstance(x, dict):
        return 1

    # Caso string
    if isinstance(x, str):
        s = x.strip()

        # intentar JSON
        try:
            obj = json.loads(s)
            if isinstance(obj, list):
                return len(obj)
            if isinstance(obj, dict):
                return 1
        except Exception:
            pass

        # intentar extraer lo de array([...])
        try:
            first = s.find('[')
            last = s.rfind(']')
            if first != -1 and last > first:
                inner = s[first:last+1]
                obj = ast.literal_eval(inner)
                if isinstance(obj, (list, tuple)):
                    return len(obj)
                if isinstance(obj, dict):
                    return 1
        except Exception:
            pass

        # intentar literal_eval directo
        try:
            obj = ast.literal_eval(s)
            if isinstance(obj, (list, tuple)):
                return len(obj)
            if isinstance(obj, dict):
                return 1
        except Exception:
            pass

        return 0

    # Caso float NaN
    try:
        if pd.isna(x):
            return 0
    except Exception:
        pass

    # fallback: usar len si existe
    try:
        return len(x)
    except Exception:
        return 0


# aplicar

#df["cant_comments"] = df["comments"].apply(lambda x: len(x) if isinstance(x, (list, tuple)) else 0)

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97898 entries, 0 to 97897
Data columns (total 10 columns):
 #   Column       Non-Null Count  Dtype              
---  ------       --------------  -----              
 0   tweet_id     97898 non-null  object             
 1   text         97898 non-null  object             
 2   title        89933 non-null  object             
 3   url          89933 non-null  object             
 4   user         97898 non-null  object             
 5   body         89933 non-null  object             
 6   created_at   97898 non-null  datetime64[ns, UTC]
 7   comments     97898 non-null  object             
 8   words_title  97898 non-null  int64              
 9   words_body   97898 non-null  int64              
dtypes: datetime64[ns, UTC](1), int64(2), object(7)
memory usage: 7.5+ MB


In [14]:
df.head(2)

,tweet_id,text,title,url,user,body,created_at,comments,words_title,words_body
0,1376941547602739202,El dólar blue cada vez más barato: cae a $ 141...,El dólar blue cada vez más barato: cayó a $ 14...,https://www.clarin.com/economia/dolar-blue-vez...,clarincom,Este martes el dólar blue retomó su tendencia ...,2021-03-30 16:57:01+00:00,"[{'created_at': '2021-03-30T16:58:04Z', 'text'...",19,801
1,1376945423273840642,Venezuela le declara la guerra a Facebook por ...,Venezuela le declara la guerra a Facebook por ...,https://www.clarin.com/tecnologia/venezuela-de...,clarincom,El presidente Nicolás Maduro le declaró la gue...,2021-03-30 17:12:25+00:00,"[{'created_at': '2021-03-30T17:14:14Z', 'text'...",14,544


In [15]:


# Agrupar por fecha y contar ids
df_counts = df.groupby(df["created_at"].dt.date)["tweet_id"].count().reset_index()
df_counts.rename(columns={"tweet_id": "count"}, inplace=True)

fig = px.line(
    df_counts,
    x="created_at",
    y="count",
    markers=True,
    title="Cantidad de IDs por fecha"
)

fig.update_layout(
    xaxis_title="Fecha",
    yaxis_title="Cantidad de IDs",
    xaxis=dict(tickangle=45)
)

fig.show()

In [16]:
# agrupar por mes
df_counts_month = df.groupby(df["created_at"].dt.to_period("M"))["tweet_id"].count().reset_index()
df_counts_month.rename(columns={"tweet_id": "count"}, inplace=True)

# opcional: convertir el periodo a string para mostrar
df_counts_month["created_at"] = df_counts_month["created_at"].astype(str)

print(df_counts_month)


   created_at  count
0     2020-02   2922
1     2020-03   5906
2     2020-04   5322
3     2020-05   4502
4     2020-06   6422
5     2020-07   4689
6     2020-08   5467
7     2020-09   5837
8     2020-10   5628
9     2020-11   5378
10    2020-12   5326
11    2021-01   5300
12    2021-02   4557
13    2021-03   4600
14    2021-04   8938
15    2021-05  12086
16    2021-06   5018


C:\Users\Ceci\AppData\Local\Temp\ipykernel_9568\1174204666.py:2: UserWarning:

Converting to PeriodArray/Index representation will drop timezone information.

